In [1]:
import os
from datetime import datetime
from pathlib import Path

import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler


In [2]:
# rutas base
REPO_ROOT = Path('.').resolve()
RUN_ID = datetime.today().strftime('%Y%m%d')
INPUT_DIR = REPO_ROOT / 'data' / 'intermediate' / '02_train_test' / RUN_ID
OUTPUT_DIR = REPO_ROOT / 'data' / 'intermediate' / '03_normalizacion' / RUN_ID

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

x_train_file = INPUT_DIR / 'X_train.parquet'
x_test_file = INPUT_DIR / 'X_test.parquet'
y_train_file = INPUT_DIR / 'y_train.parquet'
y_test_file = INPUT_DIR / 'y_test.parquet'

if not x_train_file.exists() or not x_test_file.exists():
    candidates = sorted((REPO_ROOT / 'data' / 'intermediate' / '02_train_test').glob('*/X_train.parquet'))
    if candidates:
        latest_dir = candidates[-1].parent
        x_train_file = latest_dir / 'X_train.parquet'
        x_test_file = latest_dir / 'X_test.parquet'
        y_train_file = latest_dir / 'y_train.parquet'
        y_test_file = latest_dir / 'y_test.parquet'
        print(f'Usando ultimo split encontrado: {latest_dir}')
    else:
        raise FileNotFoundError('No se encontro X_train.parquet en data/intermediate/02_train_test')

print('Rutas configuradas:')
print(f'X_TRAIN: {x_train_file}')
print(f'X_TEST: {x_test_file}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')


Usando ultimo split encontrado: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\02_train_test\20260111
Rutas configuradas:
X_TRAIN: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\02_train_test\20260111\X_train.parquet
X_TEST: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\02_train_test\20260111\X_test.parquet
OUTPUT_DIR: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\03_normalizacion\20260112


In [3]:
X_train = pd.read_parquet(x_train_file)
X_test = pd.read_parquet(x_test_file)

original_n_columns = X_train.shape[1]
print(f'Columnas originales (antes de normalizar): {original_n_columns}')

scalers = {
    'Standard': StandardScaler(),
    'MinMax': MinMaxScaler(),
    'MaxAbs': MaxAbsScaler(),
    'Robust': RobustScaler(),
    'NoNorm': None
}


Columnas originales (antes de normalizar): 545


In [4]:
# Guardar metricas
metricas = []

for name, scaler in scalers.items():
    norm_dir = OUTPUT_DIR / name
    norm_dir.mkdir(parents=True, exist_ok=True)

    if scaler:
        scaler.fit(X_train)
        X_train_norm = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
        X_test_norm = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
    else:
        X_train_norm = X_train.copy()
        X_test_norm = X_test.copy()

    X_train_norm.to_parquet(norm_dir / f'X_train_{name}.parquet')
    X_test_norm.to_parquet(norm_dir / f'X_test_{name}.parquet')

    y_train = pd.read_parquet(y_train_file)
    y_test = pd.read_parquet(y_test_file)
    y_train.to_parquet(norm_dir / f'y_train_{name}.parquet')
    y_test.to_parquet(norm_dir / f'y_test_{name}.parquet')

    n_cols_train = X_train_norm.shape[1]
    print(f'{name}: columnas despues de normalizar = {n_cols_train}')

    metricas.append({
        'normalizacion': name,
        'columnas_originales': original_n_columns,
        'columnas_resultantes': n_cols_train
    })

df_metricas = pd.DataFrame(metricas)
df_metricas.to_csv(OUTPUT_DIR / 'resumen_columnas_normalizacion.csv', index=False)
print('\nResumen guardado en: resumen_columnas_normalizacion.csv')


Standard: columnas despues de normalizar = 545
MinMax: columnas despues de normalizar = 545
MaxAbs: columnas despues de normalizar = 545
Robust: columnas despues de normalizar = 545
NoNorm: columnas despues de normalizar = 545

Resumen guardado en: resumen_columnas_normalizacion.csv
